MODIS反照率产品

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from osgeo import gdal
from pyproj import Transformer

#input
folder = r"\MODIS_Products"
output_excel =  r"\MODIS_domec.xlsx"
#SCO_U
# lat = 72.39
# lon = -27.23
#EGP
# lat = 75.63
# lon = -35.97
# #KANU
# lat = 67
# lon = -47.03
# # # QAS_L
# lat = 61.03
# lon = -46.85
# DomeC
lat = -75.1
lon = 123.3
# # kunlun
# lat = -80.428
# lon = 77.124
results = []

# 
def lonlat_to_sinu(lon, lat):
    transformer = Transformer.from_crs("EPSG:4326", 
        "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs",
        always_xy=True)
    x, y = transformer.transform(lon, lat)
    return x, y

# 
site_x, site_y = lonlat_to_sinu(lon, lat)
print(f"site WGS84: lon={lon}, lat={lat}")
print(f"site Sinu: x={site_x:.2f}m, y={site_y:.2f}m")

# 
def doy_to_date(year, doy):
    return datetime(year, 1, 1) + timedelta(days=doy - 1)

# main loop
for file in os.listdir(folder):
    if not (file.startswith("MCD43A3") and file.endswith(".hdf")):
        continue

    file_path = os.path.join(folder, file)

    #
    match = re.search(r"A(\d{4})(\d{3})", file)
    if not match:
        continue

    year = int(match.group(1))
    doy = int(match.group(2))
    date = doy_to_date(year, doy)

    try:
        ds = gdal.Open(file_path)
        subdatasets = ds.GetSubDatasets()

        # shortwave BSA
        target_sds = None
        for s in subdatasets:
            if "Albedo_BSA_shortwave" in s[0]:
                target_sds = s[0]
                break

        if target_sds is None:
            continue

        ds_sub = gdal.Open(target_sds)
        data = ds_sub.ReadAsArray().astype(float)
        gt = ds_sub.GetGeoTransform()
        nrows, ncols = data.shape

        print(f"\nprocessingh {file} -> {date.strftime('%Y-%m-%d')}")
        print(f"  data shape: {nrows} x {ncols}")
        print(f"  GeoTransform: {gt}")

        # Calculate row and column numbers
        col = int((site_x - gt[0]) / gt[1])
        row = int((site_y - gt[3]) / gt[5])

        print(f"  站点 Sinu: x={site_x:.2f}, y={site_y:.2f}")
        print(f"  计算行列号: row={row}, col={col}")

        # 
        if not (0 <= row < nrows and 0 <= col < ncols):
            print(f" 越界: 范围是 0-{nrows-1}, 0-{ncols-1}")
            continue

        # NAN
        fill_value = 32767
        data[data == fill_value] = np.nan
        data[data < 0] = np.nan
        
        # MODIS BSA 
        if np.nanmax(data) > 1:
            data = data / 1000.0

        #  3x3 window
        row_start = max(0, row - 1)
        row_end = min(nrows, row + 2)
        col_start = max(0, col - 1)
        col_end = min(ncols, col + 2)
  

        window = data[row_start:row_end, col_start:col_end]


        # valid pixel
        valid_pixels = np.sum(~np.isnan(window))

        # Retain only if at least 6 valid pixels
        if valid_pixels >= 6:
            mean_bba = np.nanmean(window)
        else:
            mean_bba = np.nan

        center_value = data[row, col]

        print(f" window shape: {window.shape}")
        print(f"  center value: {center_value:.4f}")
        print(f"  valid pixels: {valid_pixels}/{window.size}")
        print(f" mean bba: {mean_bba:.4f}")
        print("  3x3 BBA:")
        print(window)

        results.append({
            "Date": date.strftime("%Y-%m-%d"),
            "BBA_3x3": mean_bba,
            "BBA_center": center_value,
            "Valid_Pixels": valid_pixels,
            "Row": row,
            "Col": col
        })

    except Exception as e:
        print(f" error: {e}")
        import traceback
        traceback.print_exc()
        continue

# output
if results:
    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False)
    print(f"\nSaved to: {output_excel}")
    print(f"Successfully processed {len(results)} files in total")
else:
    print("\nNo data was extracted.")

四个站点反照率的时间序列

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib
import os
from matplotlib.ticker import FuncFormatter

# 
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['axes.linewidth'] = 1.0
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['axes.unicode_minus'] = False

#
start_date = pd.to_datetime("2025-04-01")
end_date   = pd.to_datetime("2025-10-01")

# 
output_dir = r"D:\Graduate\Paper\Figure"
os.makedirs(output_dir, exist_ok=True)

# four sites
sites = [
    {
        "name": "SCO_U",
        "mersi": r"D:\scou_mersi2_nocloud.xlsx",
        "insitu": r"D:\0317albedo.xlsx",
        "modis": r"D:\MODIS_BSA_SCOU_new.xlsx"
    },
    {
        "name": "EGP",
        "mersi": r"D:\EGP_MERSI2_nocloud.xlsx",
        "insitu": r"D:\EGP_albedo.xlsx",
        "modis": r"D:\MODIS_BSA_EGP_new.xlsx"
    },
    {
        "name": "KAN_U",
        "mersi": r"D:\kanu_mersi2_nocloud.xlsx",
        "insitu": r"D:\KANU_albedo.xlsx",
        "modis": r"D:\MODIS_BSA_KANU_new.xlsx"
    },
    {
        "name": "QAS_L",
        "mersi": r"D:\qasl_mersi2_nocloud.xlsx",
        "insitu": r"D:\QASL_albedo.xlsx",
        "modis": r"D:\MODIS_BSA_QASL_new.xlsx"
    }
]

# 
def y_formatter(y, pos):
    if abs(y) < 1e-6:
        return '0'
    elif abs(y - 1.0) < 1e-6:
        return '1'
    else:
        return f'{y:.2f}'

# Process four stations iteratively
for site in sites:
    print(f"\n{'='*50}")
    print(f"site: {site['name']}")
    print(f"{'='*50}")

    # -------- 1. MERSI-II --------
    df_origin = pd.read_excel(site["mersi"]).iloc[:, :2]
    df_origin.columns = ["timestamp", "BBA"]
    df_origin["timestamp"] = pd.to_datetime(df_origin["timestamp"], format="%Y%m%d_%H%M")

    # -------- 2. In-situ (AWS) --------
    df_albedo = pd.read_excel(site["insitu"])
    df_albedo.columns = ["timestamp", "albedo"]
    df_albedo["timestamp"] = pd.to_datetime(df_albedo["timestamp"])

    # -------- 3. MODIS --------
    df_modis = pd.read_excel(site["modis"])
    print(f"MODIS原始列数: {len(df_modis.columns)} | 列名: {df_modis.columns.tolist()}")
    df_modis = df_modis.iloc[:, :2]
    df_modis.columns = ["timestamp", "BBA"]
    df_modis["timestamp"] = pd.to_datetime(df_modis["timestamp"])

    # Time filtering
    df_origin = df_origin[(df_origin["timestamp"] >= start_date) & (df_origin["timestamp"] <= end_date)]
    df_albedo = df_albedo[(df_albedo["timestamp"] >= start_date) & (df_albedo["timestamp"] <= end_date)]
    df_modis  = df_modis[(df_modis["timestamp"] >= start_date) & (df_modis["timestamp"] <= end_date)]

    # plot
    fig, ax = plt.subplots(figsize=(13, 5.8))

    # MERSI-II
    ax.scatter(df_origin["timestamp"], df_origin["BBA"],
               s=70, color="#2E86DE", alpha=0.75, linewidths=0,
               label="FY-3D MERSI-II")
    
    # 
    if site['name'] == 'SCO_U':
        target_date = pd.to_datetime("2025-08-17").date()
        highlight = df_origin[df_origin["timestamp"].dt.date == target_date]
        if not highlight.empty:
            ax.scatter(highlight["timestamp"], highlight["BBA"],
                       s=400, facecolors='none', edgecolors='black', 
                       linewidth=1.5, marker='o', zorder=15)

    # MODIS
    ax.scatter(df_modis["timestamp"], df_modis["BBA"],
               s=35, marker="s", color="#27AE60", alpha=0.5,
               linewidths=0, label="MCD43A3", zorder=4)

    # In-situ
    ax.scatter(df_albedo["timestamp"], df_albedo["albedo"],
               s=70, marker="+", color="#E74C3C", edgecolor="black",
               linewidth=2, zorder=5, label="PROMICE")

    # 
    ax.set_ylim(0, 1.0)
    ax.set_xlim(start_date, end_date)
    
    # 
    ax.set_xlabel("")
    ax.set_ylabel("Broadband Albedo", fontsize=16)

    ax.yaxis.set_major_locator(plt.MultipleLocator(0.25))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)

    for label in ax.get_xticklabels():
        label.set_fontsize(16)
    for label in ax.get_yticklabels():
        label.set_fontsize(16)

    # 
    ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))

    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    
    # legend 
    ax.legend(frameon=False, fontsize=16, loc="best",
              prop={'family': 'Arial', 'size': 16})

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    plt.tight_layout()
    plt.title(site['name'], fontsize=16)
    
    # save
    save_path = os.path.join(output_dir, f"{site['name']}_albedo_300dpi.png")
    # plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"saved to: {save_path}")
    plt.show()

    print(f"Data volume: MERSI-II={len(df_origin)}, MODIS={len(df_modis)}, In-situ={len(df_albedo)}")

    # Statistical analysis (daily matching)
    mean_mersi  = df_origin["BBA"].mean()
    mean_modis  = df_modis["BBA"].mean()
    mean_insitu = df_albedo["albedo"].mean()

    # Daily scale
    df_origin["date"]  = df_origin["timestamp"].dt.date
    df_albedo["date"] = df_albedo["timestamp"].dt.date
    df_modis["date"] = df_modis["timestamp"].dt.date

    mersi_daily  = df_origin.groupby("date")["BBA"].mean().reset_index()
    insitu_daily = df_albedo.groupby("date")["albedo"].mean().reset_index()
    modis_daily  = df_modis.groupby("date")["BBA"].mean().reset_index()

    # MERSI vs In-situ
    df_daily_mersi_insitu = pd.merge(mersi_daily, insitu_daily, on="date", how="inner")
    if len(df_daily_mersi_insitu) > 0:
        mad_mersi_insitu  = np.mean(np.abs(df_daily_mersi_insitu["BBA"] - df_daily_mersi_insitu["albedo"]))
        bias_mersi_insitu = np.mean(df_daily_mersi_insitu["BBA"] - df_daily_mersi_insitu["albedo"])
        rmse_mersi_insitu = np.sqrt(np.mean((df_daily_mersi_insitu["BBA"] - df_daily_mersi_insitu["albedo"])**2))
    else:
        mad_mersi_insitu = bias_mersi_insitu = rmse_mersi_insitu = np.nan

    # MODIS vs In-situ
    df_daily_modis_insitu = pd.merge(modis_daily, insitu_daily, on="date", how="inner")
    if len(df_daily_modis_insitu) > 0:
        mad_modis_insitu  = np.mean(np.abs(df_daily_modis_insitu["BBA"] - df_daily_modis_insitu["albedo"]))
        bias_modis_insitu = np.mean(df_daily_modis_insitu["BBA"] - df_daily_modis_insitu["albedo"])
        rmse_modis_insitu = np.sqrt(np.mean((df_daily_modis_insitu["BBA"] - df_daily_modis_insitu["albedo"])**2))
    else:
        mad_modis_insitu = bias_modis_insitu = rmse_modis_insitu = np.nan

    # MERSI vs MODIS
    df_daily_mersi_modis = pd.merge(mersi_daily, modis_daily, on="date", how="inner", suffixes=("_mersi", "_modis"))
    if len(df_daily_mersi_modis) > 0:
        mad_mersi_modis  = np.mean(np.abs(df_daily_mersi_modis["BBA_mersi"] - df_daily_mersi_modis["BBA_modis"]))
        bias_mersi_modis = np.mean(df_daily_mersi_modis["BBA_mersi"] - df_daily_mersi_modis["BBA_modis"])
        rmse_mersi_modis = np.sqrt(np.mean((df_daily_mersi_modis["BBA_mersi"] - df_daily_mersi_modis["BBA_modis"])**2))
    else:
        mad_mersi_modis = bias_mersi_modis = rmse_mersi_modis = np.nan

    print("\nStatistical results")
    print(f"Time range: {start_date.date()} ~ {end_date.date()}\n")
    print(f"MERSI-II Mean BBA   : {mean_mersi:.4f}")
    print(f"MODIS Mean BBA      : {mean_modis:.4f}")
    print(f"In-situ Mean BBA    : {mean_insitu:.4f}\n")
    
    print("---- MERSI vs In-situ（daily matching）----")
    print(f"MAD  : {mad_mersi_insitu:.4f}")
    print(f"Bias : {bias_mersi_insitu:.4f}")
    print(f"RMSE : {rmse_mersi_insitu:.4f}")
    print(f"Matched days: {len(df_daily_mersi_insitu)}\n")
    
    print("---- MODIS vs In-situ（daily matching）----")
    print(f"MAD  : {mad_modis_insitu:.4f}")
    print(f"Bias : {bias_modis_insitu:.4f}")
    print(f"RMSE : {rmse_modis_insitu:.4f}")
    print(f"Matched days: {len(df_daily_modis_insitu)}\n")
    
    print("---- MERSI vs MODIS（daily matching）----")
    print(f"MAD  : {mad_mersi_modis:.4f}")
    print(f"Bias : {bias_mersi_modis:.4f}")
    print(f"RMSE : {rmse_mersi_modis:.4f}")
    print(f"Matched days: {len(df_daily_mersi_modis)}")

整体验证散点图

In [ ]:
#站点 vs MERSI-II
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
from sklearn.metrics import mean_squared_error
from matplotlib.ticker import FuncFormatter

#
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['xtick.labelsize'] = 13
matplotlib.rcParams['ytick.labelsize'] = 13
matplotlib.rcParams['axes.unicode_minus'] = False

# path
output_dir = r"D:\Figure"
os.makedirs(output_dir, exist_ok=True)

#four sites
stations = {
    'QAS_L': (r"D:\qasl_mersi2_nocloud.xlsx", r"D:\QASL_albedo.xlsx"),
    'KAN_U': (r"D:\kanu_mersi2_nocloud.xlsx", r"D:\KANU_albedo.xlsx"),
    'SCO_U': (r"D:\scou_mersi2_nocloud.xlsx", r"D:\0317albedo.xlsx"),
    'EGP':   (r"D:\EGP_MERSI2_nocloud.xlsx", r"D:\EGP_albedo.xlsx")
}

#
all_data = []

# 
for name, (sat_path, obs_path) in stations.items():
    
    sat = pd.read_excel(sat_path)
    obs = pd.read_excel(obs_path)
    
    # time
    sat['date'] = sat['timestamp'].astype(str).str.split('_').str[0]
    sat['date'] = pd.to_datetime(sat['date'], format='%Y%m%d')
    
    # Daily average
    sat_daily = sat.groupby('date')['BBA'].mean().reset_index()
    sat_daily.rename(columns={'BBA': 'sat_albedo'}, inplace=True)
    
    # measurement
    obs['date'] = pd.to_datetime(obs['time'])
    obs.rename(columns={'albedo': 'obs_albedo'}, inplace=True)
    
    # matching
    df = pd.merge(sat_daily, obs[['date','obs_albedo']], on='date', how='inner')
    df = df.dropna()
    df = df[(df['sat_albedo'] > 0) & (df['obs_albedo'] > 0)]
    
    df['station'] = name
    
    all_data.append(df)

# 
df_all = pd.concat(all_data, ignore_index=True)

#
x = df_all['obs_albedo']
y = df_all['sat_albedo']

N = len(df_all)
bias = np.mean(y - x)
rmse = np.sqrt(mean_squared_error(x, y))
r = np.corrcoef(x, y)[0,1]
r2 = r**2
slope, intercept = np.polyfit(x, y, 1)

#plot
plt.figure(figsize=(6,6), dpi=300)

# 
colors = {
    'QAS_L': 'red',
    'KAN_U': 'blue',
    'SCO_U': 'green',
    'EGP': 'orange'
}

for station in df_all['station'].unique():
    d = df_all[df_all['station'] == station]
    plt.scatter(d['obs_albedo'], d['sat_albedo'],
                s=35, alpha=0.75,
                label=station,
                color=colors[station],
                edgecolor='none')

#
plt.plot([0,1], [0,1], 'k-', linewidth=1)

# 
xx = np.linspace(0,1,100)
plt.plot(xx, slope*xx + intercept, 'k--', linewidth=1)

# 
plt.xlim(0,1)
plt.ylim(0,1)
plt.xlabel('PROMICE AWS Albedo', fontsize=13)
plt.ylabel('FY-3D MERSI-II Albedo', fontsize=13)

# 
plt.legend(frameon=True, fontsize=13)

# 
textstr = '\n'.join((
    f'N = {N}',
    f'Bias = {bias:.3f}',
    f'RMSE = {rmse:.3f}',
    f'R² = {r2:.3f}',
    f'y = {slope:.3f}x {intercept:+.3f}'
))

plt.text(0.05, 0.75, textstr,
         fontsize=13,
         bbox=dict(facecolor='white', alpha=0.8))

# 
plt.grid(True, linestyle='--', alpha=0.3)

# 
def format_x(x, pos):
    if abs(x) < 1e-6:
        return '0'
    elif abs(x-1) < 1e-6:
        return '1'
    else:
        return f'{x:.1f}'

def format_y(y, pos):
    if abs(y) < 1e-6:
        return ''
    elif abs(y-1) < 1e-6:
        return '1'
    else:
        return f'{y:.1f}'

ax = plt.gca()
ax.xaxis.set_major_formatter(FuncFormatter(format_x))
ax.yaxis.set_major_formatter(FuncFormatter(format_y))
ax.tick_params(direction='in', which='both')

plt.tight_layout()

# 
save_path = os.path.join(output_dir, "multi_station_validation.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"图片已保存: {save_path}")

plt.show()

In [ ]:
# MERSI2 VS MCD43A3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
from sklearn.metrics import mean_squared_error
from matplotlib.ticker import FuncFormatter


#

matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['xtick.labelsize'] = 13
matplotlib.rcParams['ytick.labelsize'] = 13
matplotlib.rcParams['axes.unicode_minus'] = False


# ============================================================
# 输出路径
# ============================================================

output_dir = r"D:\Figure"

os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 站点数据
# ============================================================

stations = {

    'QAS_L': (
        r"D:\qasl_mersi2_nocloud.xlsx",
        r"D:\MODIS_BSA_QASL_new.xlsx"
    ),

    'KAN_U': (
        r"D:\kanu_mersi2_nocloud.xlsx",
        r"D:\MODIS_BSA_KANU_new.xlsx"
    ),

    'SCO_U': (
        r"D:\scou_mersi2_nocloud.xlsx",
        r"D:\MODIS_BSA_SCOU_new.xlsx"
    ),

    'EGP': (
        r"D:\EGP_MERSI2_nocloud.xlsx",
        r"D:\MODIS_BSA_EGP_new.xlsx"
    )

}


#

all_data = []


for name, (sat_path, obs_path) in stations.items():

    sat = pd.read_excel(sat_path)
    obs = pd.read_excel(obs_path)


    # ------
    # MERSI2


    sat['timestamp'] = pd.to_datetime(
        sat['timestamp'],
        format="%Y%m%d_%H%M"
    )

    sat['date'] = sat['timestamp'].dt.date

    sat['date'] = pd.to_datetime(
        sat['date']
    )

    sat_daily = (
        sat.groupby('date')['BBA']
        .mean()
        .reset_index()
    )

    sat_daily.rename(
        columns={
            'BBA': 'sat_albedo'
        },
        inplace=True
    )


    # ----
    # MODIS


    obs = obs.iloc[:, :2]

    obs.columns = [
        'date_raw',
        'modis_albedo'
    ]

    obs['date'] = pd.to_datetime(
        obs['date_raw']
    )


    # 

    df = pd.merge(
        sat_daily,
        obs[['date', 'modis_albedo']],
        on='date',
        how='inner'
    )

    df = df.dropna()

    df = df[
        (df['sat_albedo'] > 0)
        &
        (df['modis_albedo'] > 0)
    ]

    df['station'] = name

    all_data.append(df)


# 

df_all = pd.concat(
    all_data,
    ignore_index=True
)



# 整体统计


x = df_all['modis_albedo']

y = df_all['sat_albedo']

N = len(df_all)


# Bias
bias = np.mean(
    y - x
)


# MAE
mae = np.mean(
    np.abs(y - x)
)


# RMSE
rmse = np.sqrt(
    mean_squared_error(x, y)
)


# Correlation coefficient
r = np.corrcoef(
    x,
    y
)[0, 1]


# R²
r2 = r ** 2


# Linear regression
slope, intercept = np.polyfit(
    x,
    y,
    1
)


# 
plt.figure(
    figsize=(6, 6),
    dpi=300
)


# 

colors = {

    'QAS_L': 'red',

    'KAN_U': 'blue',

    'SCO_U': 'green',

    'EGP': 'orange'

}


# 

for station in df_all['station'].unique():

    d = df_all[
        df_all['station'] == station
    ]

    plt.scatter(
        d['modis_albedo'],
        d['sat_albedo'],
        s=35,
        alpha=0.75,
        label=station,
        color=colors[station],
        edgecolor='none'
    )


#

plt.plot(
    [0, 1],
    [0, 1],
    'k-',
    linewidth=1
)


# 

xx = np.linspace(
    0,
    1,
    100
)

plt.plot(
    xx,
    slope * xx + intercept,
    'k--',
    linewidth=1
)


# 

plt.xlim(
    0,
    1
)

plt.ylim(
    0,
    1
)


# 

plt.xlabel(
    'MCD43A3',
    fontsize=13
)

plt.ylabel(
    'FY-3D MERSI-II Albedo',
    fontsize=13
)


#

plt.legend(
    frameon=True,
    fontsize=13
)


#

textstr = '\n'.join((

    f'N = {N}',

    f'Bias = {bias:.3f}',

    f'RMSE = {rmse:.3f}',

    f'R² = {r2:.3f}',

    f'y = {slope:.3f}x {intercept:+.3f}'

))


plt.text(
    0.05,
    0.75,
    textstr,
    fontsize=13,
    bbox=dict(
        facecolor='white',
        alpha=0.8
    )
)


# 

plt.grid(
    True,
    linestyle='--',
    alpha=0.3
)


# 

def format_x(x, pos):

    if abs(x) < 1e-6:

        return '0'

    elif abs(x - 1) < 1e-6:

        return '1'

    else:

        return f'{x:.1f}'


# 

def format_y(y, pos):

    if abs(y) < 1e-6:

        return ''

    elif abs(y - 1) < 1e-6:

        return '1'

    else:

        return f'{y:.1f}'


# 

ax = plt.gca()


ax.xaxis.set_major_formatter(
    FuncFormatter(format_x)
)

ax.yaxis.set_major_formatter(
    FuncFormatter(format_y)
)


# 

ax.tick_params(
    direction='in',
    which='both'
)


# 

plt.tight_layout()


# 

save_path = os.path.join(
    output_dir,
    "MERSI2_vs_MCD43A3.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)

print(
    f"saaved to: {save_path}"
)


# 

plt.show()

MERSI2和OLCI空间对比图

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.gridspec as gridspec
import os
from rasterio.windows import from_bounds
from pyproj import Transformer
from matplotlib.ticker import MultipleLocator, FuncFormatter

# 
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['xtick.labelsize'] = 13
matplotlib.rcParams['ytick.labelsize'] = 13
matplotlib.rcParams['axes.labelsize'] = 13
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.unicode_minus'] = False

# 
output_dir = r"D:\Figure"
os.makedirs(output_dir, exist_ok=True)

# 
file_mersi = r"D:\BBA_FY3D_MERSI_GBAL_L1_20250416_1505_subset.tif"
file_olci  = r"D:\albedo_bb_planar_sw.tif"

# Target extent
lon_min, lon_max = -28, -24
lat_min, lat_max = 74.5, 76

#
def read_crop(fp, lon_min, lon_max, lat_min, lat_max):
    with rasterio.open(fp) as src:
        crs = src.crs

        if crs.to_string() != "EPSG:4326":
            transformer = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
            x1, y1 = transformer.transform(lon_min, lat_min)
            x2, y2 = transformer.transform(lon_max, lat_max)

            window = from_bounds(min(x1,x2), min(y1,y2),
                                 max(x1,x2), max(y1,y2),
                                 src.transform)
        else:
            window = from_bounds(lon_min, lat_min,
                                 lon_max, lat_max,
                                 src.transform)

        data = src.read(1, window=window).astype(float)
        transform = src.window_transform(window)

        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        return data, transform

# 
data1, trans1 = read_crop(file_mersi, lon_min, lon_max, lat_min, lat_max)
data2, trans2 = read_crop(file_olci,  lon_min, lon_max, lat_min, lat_max)

# 
def cbar_formatter(x, pos):
    if abs(x) < 1e-6:
        return '0'
    elif abs(x - 1.0) < 1e-6:
        return '1'
    else:
        return f'{x:.2f}'

# plot
fig = plt.figure(figsize=(10, 9))

gs = gridspec.GridSpec(
    2, 1,
    height_ratios=[1, 1],
    hspace=0.2
)

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])

# MERSI
im1 = ax0.imshow(
    data1,
    cmap='viridis',
    vmin=0.75, vmax=0.95,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin='upper'
)

ax0.set_title("FY-3D MERSI-II (15:05 UTC)")
ax0.set_xlabel("Longitude, degrees")
ax0.set_ylabel("Latitude, degrees")

# OLCI
im2 = ax1.imshow(
    data2,
    cmap='viridis',
    vmin=0.75, vmax=0.95,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin='upper'
)

ax1.set_title("S3 OLCI (13:33 UTC)")
ax1.set_xlabel("Longitude, degrees")
ax1.set_ylabel("Latitude, degrees")

#
for ax in [ax0, ax1]:
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(0.25))
    ax.tick_params(direction='in', which='both')

# 
cbar = fig.colorbar(
    im1,
    ax=[ax0, ax1],
    orientation='vertical',
    fraction=0.03,
    pad=0.04
)

cbar.set_label("BBA")
cbar.ax.tick_params(direction='in', which='both')
cbar.ax.yaxis.set_major_formatter(FuncFormatter(cbar_formatter))

# # 
save_path = os.path.join(output_dir, "MERSI_OLCI_comparison_spatial.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"saved to: {save_path}")

plt.show()

直方图

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

from rasterio.windows import from_bounds
from pyproj import Transformer
from rasterio.warp import reproject, Resampling
from rasterio.transform import xy
from matplotlib.ticker import MultipleLocator
from matplotlib.colors import LinearSegmentedColormap


# 

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 0.6

plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

plt.rcParams['xtick.major.width'] = 0.6
plt.rcParams['ytick.major.width'] = 0.6
plt.rcParams['xtick.minor.width'] = 0.4
plt.rcParams['ytick.minor.width'] = 0.4


# 

file_mersi = (
    r"D:\BBA_FY3D_MERSI_GBAL_L1_20250416_1505_subset.tif"
)

file_olci = (
    r"D:\albedo_bb_planar_sw.tif"
)

lon_min, lon_max = -28, -24
lat_min, lat_max = 74.5, 76


def read_crop(fp, lon_min, lon_max, lat_min, lat_max):

    with rasterio.open(fp) as src:

        crs = src.crs

        if crs.to_string() != "EPSG:4326":

            transformer = Transformer.from_crs(
                "EPSG:4326",
                crs,
                always_xy=True
            )

            x1, y1 = transformer.transform(
                lon_min,
                lat_min
            )

            x2, y2 = transformer.transform(
                lon_max,
                lat_max
            )

            window = from_bounds(
                min(x1, x2),
                min(y1, y2),
                max(x1, x2),
                max(y1, y2),
                src.transform
            )

        else:

            window = from_bounds(
                lon_min,
                lat_min,
                lon_max,
                lat_max,
                src.transform
            )

        data = src.read(
            1,
            window=window
        ).astype(float)

        transform = src.window_transform(window)

        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        return data, transform, crs



data1, trans1, crs1 = read_crop(
    file_mersi,
    lon_min,
    lon_max,
    lat_min,
    lat_max
)

data2, trans2, crs2 = read_crop(
    file_olci,
    lon_min,
    lon_max,
    lat_min,
    lat_max
)



# 重采样 OLCI → MERSI

data2_resampled = np.empty_like(
    data1,
    dtype=np.float32
)

reproject(
    source=data2,
    destination=data2_resampled,
    src_transform=trans2,
    src_crs=crs2,
    dst_transform=trans1,
    dst_crs=crs1,
    resampling=Resampling.bilinear
)



rows, cols = data1.shape

row_idx, col_idx = np.mgrid[
    0:rows,
    0:cols
]

xs, ys = xy(
    trans1,
    row_idx,
    col_idx
)

if crs1.to_string() != "EPSG:4326":

    transformer = Transformer.from_crs(
        crs1,
        "EPSG:4326",
        always_xy=True
    )

    lons, lats = transformer.transform(
        xs,
        ys
    )

else:

    lons, lats = xs, ys

lat_array = np.array(lats)


# ============================================================
# 配对像素

x = data2_resampled.flatten()      # OLCI
y = data1.flatten()                # MERSI
lat_flat = lat_array.flatten()

mask = (
    ~np.isnan(x)
    &
    ~np.isnan(y)
)

x = x[mask]
y = y[mask]
lat_flat = lat_flat[mask]

N = len(x)

print(
    "Number of paired pixels:",
    N
)



# 差值统计

diff = y - x

mean_diff = np.mean(diff)
median_diff = np.median(diff)
std_diff = np.std(diff)

print(f"Mean   = {mean_diff:.4f}")
print(f"Median = {median_diff:.4f}")
print(f"Std    = {std_diff:.4f}")

# MERSI / OLCI 

mean_mersi = np.mean(y)
median_mersi = np.median(y)
std_mersi = np.std(y)

mean_olci = np.mean(x)
median_olci = np.median(x)
std_olci = np.std(x)




density_cmap = LinearSegmentedColormap.from_list(
    'density_cmap',
    [
        '#DCE6F1',   # very light blue-gray
        '#A9BED3',
        '#6E91AF',
        '#456F8F',
        '#2F536F'    # dark blue-gray
    ]
)



COLOR_HIST = '#7E9FB8'
COLOR_HIST_EDGE = '#496B84'



COLOR_MEAN = '#C44536'
COLOR_MEDIAN = '#E8A838'

COLOR_ZERO = '#333333'
COLOR_SHADE = '#C44536'

COLOR_TEXT = '#2C3E50'

COLOR_BOX_BG = '#F5F0E8'

# 组合图

fig = plt.figure(
    figsize=(8.0, 7.0),
    dpi=300
)



# GridSpec


gs = fig.add_gridspec(
    nrows=2,
    ncols=2,
    width_ratios=[1.45, 0.45],
    height_ratios=[0.45, 1.45],

    hspace=0.15,
    wspace=0.15
)


# (a) OLCI 

ax_top = fig.add_subplot(
    gs[0, 0]
)


# OLCI histogram


weights_olci = (
    np.ones_like(x)
    * 100.0
    / N
)

ax_top.hist(
    x,
    bins=40,
    range=(0.70, 0.90),
    weights=weights_olci,
    color=COLOR_HIST,
    edgecolor=COLOR_HIST_EDGE,
    linewidth=0.4,
    alpha=0.85
)




ax_top.set_xlim(
    0.70,
    0.90
)

ax_top.set_ylim(
    0,
    30
)


ax_top.set_xticks(
    [
        0.70,
        0.75,
        0.80,
        0.85,
        0.90
    ]
)

ax_top.set_xlabel(
    'OLCI Albedo',
    fontsize=9,
    labelpad=3
)



ax_top.set_ylabel(
    'Percentage (%)',
    fontsize=9,
    labelpad=4
)

ax_top.set_yticks(
    [
        0,
        10,
        20,
        30
    ]
)


# 

ax_top.tick_params(
    axis='both',
    direction='in',
    which='both',
    width=0.6,
    length=3,
    labelsize=8
)


#

ax_top.set_title(
    'OLCI',
    fontsize=10,
    pad=4
)


#

olci_text = (
    f'μ = {mean_olci:.5f}\n'
    f'Median = {median_olci:.5f}\n'
    f'σ = {std_olci:.5f}'
)

ax_top.text(
    0.03,
    0.90,
    olci_text,
    transform=ax_top.transAxes,
    fontsize=7.5,
    color=COLOR_TEXT,
    ha='left',
    va='top',
    multialignment='left',
    bbox=dict(
        boxstyle='round,pad=0.25',
        facecolor='white',
        edgecolor='#B8B0A0',
        linewidth=0.5,
        alpha=0.90
    )
)


# 

ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)



# (b) MERSI-II vs OLCI 

ax = fig.add_subplot(
    gs[1, 0]
)



# 密度计算

# x = OLCI
# y = MERSI-II


density_bins = 80

density, x_edges, y_edges = np.histogram2d(
    x,
    y,
    bins=density_bins,
    range=[
        [0.70, 0.90],
        [0.70, 0.90]
    ]
)



x_bin = np.clip(
    np.digitize(x, x_edges) - 1,
    0,
    density_bins - 1
)

y_bin = np.clip(
    np.digitize(y, y_edges) - 1,
    0,
    density_bins - 1
)

point_density = density[
    x_bin,
    y_bin
]



sort_idx = np.argsort(
    point_density
)


# 密度散点图


scatter = ax.scatter(
    x[sort_idx],
    y[sort_idx],
    s=9,
    c=point_density[sort_idx],
    cmap=density_cmap,
    alpha=0.70,
    edgecolors='none'
)


# 线性拟合

coef = np.polyfit(
    x,
    y,
    1
)

slope = coef[0]
intercept = coef[1]


# ============================================================
# 拟合值
# ============================================================

y_fit = (
    slope * x
    + intercept
)


# ============================================================
# R²
# ============================================================

ss_res = np.sum(
    (y - y_fit) ** 2
)

ss_tot = np.sum(
    (y - np.mean(y)) ** 2
)

r2 = 1 - ss_res / ss_tot


# ============================================================
# RMSE
# ============================================================

rmse = np.sqrt(
    np.mean(
        (y - x) ** 2
    )
)


# ============================================================
# Bias
# ============================================================

bias = np.mean(
    y - x
)


# ============================================================
# 1:1 line
# ============================================================

ax.plot(
    [0.70, 0.90],
    [0.70, 0.90],
    color=COLOR_ZERO,
    linestyle='--',
    linewidth=0.9
)


# ============================================================
# Linear fit
# ============================================================

xx = np.linspace(
    0.70,
    0.90,
    200
)

yy = (
    slope * xx
    + intercept
)

ax.plot(
    xx,
    yy,
    color=COLOR_MEAN,
    linewidth=1.2
)


# ============================================================
# 主图坐标范围
# ============================================================

ax.set_xlim(
    0.70,
    0.90
)

ax.set_ylim(
    0.70,
    0.90
)


# ============================================================
# 主图坐标轴
# ============================================================

ax.set_xlabel(
    'OLCI Albedo',
    fontsize=11,
    labelpad=5
)

ax.set_ylabel(
    'MERSI-II Albedo',
    fontsize=11,
    labelpad=5
)


# ============================================================
# 主图刻度
# ============================================================

ax.set_xticks(
    [
        0.70,
        0.75,
        0.80,
        0.85,
        0.90
    ]
)

ax.set_yticks(
    [
        0.70,
        0.75,
        0.80,
        0.85,
        0.90
    ]
)

# 
ax.set_xticklabels(['', '0.75', '0.80', '0.85', '0.90'])
ax.tick_params(
    axis='both',
    direction='in',
    which='both',
    width=0.6,
    length=3,
    labelsize=9
)




textstr = (
    f'R² = {r2:.3f}\n'
    f'RMSE = {rmse:.4f}\n'
    f'Bias = {bias:.4f}\n'
    f'N = {N:,}'
)

ax.text(
    0.72,
    0.05,
    textstr,
    transform=ax.transAxes,
    fontsize=8.5,
    ha='left',
    va='bottom',
    multialignment='left',
    color=COLOR_TEXT,
    bbox=dict(
        boxstyle='round,pad=0.3',
        facecolor='white',
        edgecolor='#B8B0A0',
        linewidth=0.5,
        alpha=0.90
    )
)



equation_x = 0.835

equation_y = (
    slope * equation_x
    + intercept
)


equation_text = (
    f'y = {slope:.3f}x '
    f'{intercept:+.3f}'
)




angle = np.degrees(
    np.arctan(slope)
)


ax.text(
    equation_x,
    equation_y,
    equation_text,
    fontsize=11,
    color=COLOR_MEAN,
    ha='left',
    va='bottom',

    rotation=angle,
    rotation_mode='anchor'
)


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


# ============================================================
# (c) MERSI-II 
ax_right = fig.add_subplot(
    gs[1, 1]
)


# ============================================================
# MERSI-II histogram


weights_mersi = (
    np.ones_like(y)
    * 100.0
    / N
)

ax_right.hist(
    y,
    bins=40,
    range=(0.70, 0.90),
    weights=weights_mersi,
    orientation='horizontal',

    # 与主图密度色带配套
    color=COLOR_HIST,
    edgecolor=COLOR_HIST_EDGE,

    linewidth=0.4,
    alpha=0.85
)


# 

ax_right.set_xlim(
    0,
    30
)

ax_right.set_ylim(
    0.70,
    0.90
)


#

ax_right.set_xlabel(
    'Percentage (%)',
    fontsize=9,
    labelpad=3
)

ax_right.set_xticks(
    [
        0,
        10,
        20,
        30
    ]
)


# 
ax_right.set_yticks(
    [
        0.70,
        0.75,
        0.80,
        0.85,
        0.90
    ]
)

ax_right.set_ylabel(
    'MERSI-II Albedo',
    fontsize=9,
    labelpad=4
)


# 

ax_right.tick_params(
    axis='both',
    direction='in',
    which='both',
    width=0.6,
    length=3,
    labelsize=8
)


# 

ax_right.set_title(
    'MERSI-II',
    fontsize=10,
    pad=4
)


# 

mersi_text = (
    f'μ = {mean_mersi:.5f}\n'
    f'Median = {median_mersi:.5f}\n'
    f'σ = {std_mersi:.5f}'
)

ax_right.text(
    0.03,
    0.90,
    mersi_text,
    transform=ax_right.transAxes,
    fontsize=7.5,
    color=COLOR_TEXT,
    ha='left',
    va='top',
    multialignment='left',
    bbox=dict(
        boxstyle='round,pad=0.25',
        facecolor='white',
        edgecolor='#B8B0A0',
        linewidth=0.5,
        alpha=0.90
    )
)


#

ax_right.spines['top'].set_visible(False)
ax_right.spines['right'].set_visible(False)


# 

plt.subplots_adjust(
    left=0.10,
    right=0.96,
    bottom=0.09,
    top=0.95,

    wspace=0.15,
    hspace=0.15
)


# 

for spine in ax.spines.values():

    spine.set_visible(True)
    spine.set_linewidth(0.6)
    spine.set_color('black')


# 

# plt.savefig(
#     r"D:\Figure",
#     dpi=600,
#     bbox_inches='tight'
# )


plt.show()

南极反照率时间序列图

In [ ]:
# Dome A
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib
import os
from matplotlib.ticker import FuncFormatter

# 
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['axes.linewidth'] = 1.0
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['xtick.labelsize'] = 13
matplotlib.rcParams['ytick.labelsize'] = 13
matplotlib.rcParams['axes.labelsize'] = 13
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.unicode_minus'] = False

# 
output_dir = r"\Figure"
os.makedirs(output_dir, exist_ok=True)

# # 1. MERSI-II
file_path_origin = r"D:\domeA_mersi2_nocloud.xlsx"
df_origin = pd.read_excel(file_path_origin)
df_origin = df_origin.iloc[:, :2]
df_origin.columns = ["timestamp", "BBA"]
df_origin["timestamp"] = pd.to_datetime(df_origin["timestamp"], format="%Y%m%d_%H%M")

# 2.
file_path_albedo = r"D:\DomeA_albedo.xlsx"
df_albedo = pd.read_excel(file_path_albedo)
df_albedo.columns = ["timestamp", "albedo"]
df_albedo["timestamp"] = pd.to_datetime(df_albedo["timestamp"], format="%Y%m%d")

# 3. MODIS
file_path_modis = r"D:\MODIS_BSA_DomeA.xlsx"
df_modis = pd.read_excel(file_path_modis)

print(f"MODIS原始列数: {len(df_modis.columns)}")
print(f"MODIS列名: {df_modis.columns.tolist()}")

#
df_modis = df_modis.iloc[:, :2]
df_modis.columns = ["timestamp", "BBA"]
df_modis["timestamp"] = pd.to_datetime(df_modis["timestamp"])
df_modis = df_modis.iloc[:, :2]
df_modis.columns = ["timestamp", "BBA"]
df_modis["timestamp"] = pd.to_datetime(df_modis["timestamp"])

# time range
start_date = pd.to_datetime("2022-12-01")
end_date = pd.to_datetime("2023-03-01")

df_origin = df_origin[(df_origin["timestamp"] >= start_date) & (df_origin["timestamp"] <= end_date)]
df_albedo = df_albedo[(df_albedo["timestamp"] >= start_date) & (df_albedo["timestamp"] <= end_date)]
df_modis = df_modis[(df_modis["timestamp"] >= start_date) & (df_modis["timestamp"] <= end_date)]

# 
def y_formatter(y, pos):
    if abs(y) < 1e-6:
        return '0'
    elif abs(y - 1.0) < 1e-6:
        return '1'
    else:
        return f'{y:.2f}'

# plot
fig, ax = plt.subplots(figsize=(13, 5.8))

# -------- MERSI-II --------
ax.scatter(df_origin["timestamp"], df_origin["BBA"],
           s=70,
           color="#2E86DE",
           alpha=0.75,
           linewidths=0,
           label="FY-3D MERSI-II")

# -------- MODIS --------
ax.scatter(df_modis["timestamp"], df_modis["BBA"],
           s=40,
           marker="s",
           color="#27AE60",
           alpha=0.5,
           linewidths=0,
           label="MCD43A3",
           zorder=4)

# -------- In-situ Albedo --------
ax.scatter(df_albedo["timestamp"], df_albedo["albedo"],
           s=70,
           marker="+",
           color="#E74C3C",
           edgecolor="black",
           linewidth=2,
           zorder=5,
           label="Jia et al., 2025")

# 
ax.set_ylim(0, 1.0)
ax.set_xlim(start_date, end_date)

# 
ax.set_xlabel("")
ax.set_ylabel("Broadband Albedo", fontsize=16)

ax.yaxis.set_major_locator(plt.MultipleLocator(0.25))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

for label in ax.get_xticklabels():
    label.set_fontsize(16)
for label in ax.get_yticklabels():
    label.set_fontsize(16)

# 
ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))

ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

# legend
ax.legend(frameon=False, fontsize=16, loc="best",
          prop={'family': 'Arial', 'size': 16})

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.tight_layout()
plt.title('Dome A', fontsize=16)

# 
save_path = os.path.join(output_dir, "DomeA_albedo_300dpi.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"saved to: {save_path}")

plt.show()

print(f"\nData volume: MERSI-II={len(df_origin)}, MODIS={len(df_modis)}, In-situ={len(df_albedo)}")


# 

stat_start = pd.to_datetime("2022-12-01")
stat_end   = pd.to_datetime("2023-03-01")

df_origin_s = df_origin[(df_origin["timestamp"] >= stat_start) & (df_origin["timestamp"] <= stat_end)].copy()
df_modis_s  = df_modis[(df_modis["timestamp"] >= stat_start) & (df_modis["timestamp"] <= stat_end)].copy()
df_albedo_s = df_albedo[(df_albedo["timestamp"] >= stat_start) & (df_albedo["timestamp"] <= stat_end)].copy()

# average
mean_mersi  = df_origin_s["BBA"].mean()
mean_modis  = df_modis_s["BBA"].mean()
mean_insitu = df_albedo_s["albedo"].mean()

# Daily-scale processing

df_origin_s["date"] = df_origin_s["timestamp"].dt.date
df_albedo_s["date"] = df_albedo_s["timestamp"].dt.date
df_modis_s["date"] = df_modis_s["timestamp"].dt.date

mersi_daily  = df_origin_s.groupby("date")["BBA"].mean().reset_index()
insitu_daily = df_albedo_s.groupby("date")["albedo"].mean().reset_index()
modis_daily  = df_modis_s.groupby("date")["BBA"].mean().reset_index()


#  MERSI vs In-situ Statistical metrics

df_daily_mersi_insitu = pd.merge(mersi_daily, insitu_daily, on="date", how="inner")

if len(df_daily_mersi_insitu) > 0:
    mad_mersi_insitu  = np.mean(np.abs(df_daily_mersi_insitu["BBA"] - df_daily_mersi_insitu["albedo"]))
    bias_mersi_insitu = np.mean(df_daily_mersi_insitu["BBA"] - df_daily_mersi_insitu["albedo"])
    rmse_mersi_insitu = np.sqrt(np.mean((df_daily_mersi_insitu["BBA"] - df_daily_mersi_insitu["albedo"])**2))
    
    ss_res = np.sum((df_daily_mersi_insitu["BBA"] - df_daily_mersi_insitu["albedo"])**2)
    ss_tot = np.sum((df_daily_mersi_insitu["albedo"] - np.mean(df_daily_mersi_insitu["albedo"]))**2)
    r2_mersi_insitu = 1 - ss_res / ss_tot
else:
    mad_mersi_insitu = bias_mersi_insitu = rmse_mersi_insitu = r2_mersi_insitu = np.nan

#  MODIS vs In-situ Statistical metrics

df_daily_modis_insitu = pd.merge(modis_daily, insitu_daily, on="date", how="inner")

if len(df_daily_modis_insitu) > 0:
    mad_modis_insitu  = np.mean(np.abs(df_daily_modis_insitu["BBA"] - df_daily_modis_insitu["albedo"]))
    bias_modis_insitu = np.mean(df_daily_modis_insitu["BBA"] - df_daily_modis_insitu["albedo"])
    rmse_modis_insitu = np.sqrt(np.mean((df_daily_modis_insitu["BBA"] - df_daily_modis_insitu["albedo"])**2))
    
    ss_res = np.sum((df_daily_modis_insitu["BBA"] - df_daily_modis_insitu["albedo"])**2)
    ss_tot = np.sum((df_daily_modis_insitu["albedo"] - np.mean(df_daily_modis_insitu["albedo"]))**2)
    r2_modis_insitu = 1 - ss_res / ss_tot
else:
    mad_modis_insitu = bias_modis_insitu = rmse_modis_insitu = r2_modis_insitu = np.nan

#  MERSI vs MODIS Statistical metrics

df_daily_mersi_modis = pd.merge(mersi_daily, modis_daily, on="date", how="inner")

if len(df_daily_mersi_modis) > 0:
    mad_mersi_modis  = np.mean(np.abs(df_daily_mersi_modis["BBA_x"] - df_daily_mersi_modis["BBA_y"]))
    bias_mersi_modis = np.mean(df_daily_mersi_modis["BBA_x"] - df_daily_mersi_modis["BBA_y"])
    rmse_mersi_modis = np.sqrt(np.mean((df_daily_mersi_modis["BBA_x"] - df_daily_mersi_modis["BBA_y"])**2))
    
    ss_res = np.sum((df_daily_mersi_modis["BBA_x"] - df_daily_mersi_modis["BBA_y"])**2)
    ss_tot = np.sum((df_daily_mersi_modis["BBA_y"] - np.mean(df_daily_mersi_modis["BBA_y"]))**2)
    r2_mersi_modis = 1 - ss_res / ss_tot
else:
    mad_mersi_modis = bias_mersi_modis = rmse_mersi_modis = r2_mersi_modis = np.nan

# 
print("\n================ Statistical results =================")
print("time range: 2022-12-01 ~ 2023-03-01\n")

print(f"MERSI-II Mean BBA   : {mean_mersi:.4f}")
print(f"MODIS Mean BBA      : {mean_modis:.4f}")
print(f"In-situ Mean BBA    : {mean_insitu:.4f}\n")

print("---- MERSI vs In-situ（Daily matching）----")
print(f"samples N   : {len(df_daily_mersi_insitu)}")
print(f"MAD        : {mad_mersi_insitu:.4f}")
print(f"Bias       : {bias_mersi_insitu:.4f}")
print(f"RMSE       : {rmse_mersi_insitu:.4f}")
print(f"R2         : {r2_mersi_insitu:.4f}\n")

print("---- MODIS vs In-situ（Daily matching）----")
print(f"samples N   : {len(df_daily_modis_insitu)}")
print(f"MAD        : {mad_modis_insitu:.4f}")
print(f"Bias       : {bias_modis_insitu:.4f}")
print(f"RMSE       : {rmse_modis_insitu:.4f}")
print(f"R2         : {r2_modis_insitu:.4f}\n")

print("---- MERSI vs MODIS（Daily matching）----")
print(f"samples N   : {len(df_daily_mersi_modis)}")
print(f"MAD        : {mad_mersi_modis:.4f}")
print(f"Bias       : {bias_mersi_modis:.4f}")
print(f"RMSE       : {rmse_mersi_modis:.4f}")
print(f"R2         : {r2_mersi_modis:.4f}")

In [ ]:
# Dome C
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib
import os
from matplotlib.ticker import FuncFormatter

# 
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['axes.linewidth'] = 1.0
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['xtick.labelsize'] = 13
matplotlib.rcParams['ytick.labelsize'] = 13
matplotlib.rcParams['axes.labelsize'] = 13
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.unicode_minus'] = False

# 
output_dir = r"\Figure"
os.makedirs(output_dir, exist_ok=True)



# MERSI-II
file_path_origin = r"\domec_mersi2_nocloud.xlsx"
df_origin = pd.read_excel(file_path_origin)
df_origin = df_origin.iloc[:, :2]
df_origin.columns = ["timestamp", "BBA"]
df_origin["timestamp"] = pd.to_datetime(df_origin["timestamp"], format="%Y%m%d_%H%M")

#  MODIS
file_path_modis = r"\MODIS_domec_new.xlsx"
df_modis = pd.read_excel(file_path_modis)

print(f"MODIS Original number of columns: {len(df_modis.columns)}")
print(f"MODIS Column names: {df_modis.columns.tolist()}")

df_modis = df_modis.iloc[:, :2]
df_modis.columns = ["timestamp", "BBA"]
df_modis["timestamp"] = pd.to_datetime(df_modis["timestamp"])

# time range
start_date = pd.to_datetime("2025-12-01")
end_date = pd.to_datetime("2026-03-01")

df_origin = df_origin[(df_origin["timestamp"] >= start_date) & (df_origin["timestamp"] <= end_date)]
df_modis = df_modis[(df_modis["timestamp"] >= start_date) & (df_modis["timestamp"] <= end_date)]

# 
def y_formatter(y, pos):
    if abs(y) < 1e-6:
        return '0'
    elif abs(y - 1.0) < 1e-6:
        return '1'
    else:
        return f'{y:.2f}'

# 
fig, ax = plt.subplots(figsize=(13, 5.8))

# -------- MERSI-II --------
ax.scatter(df_origin["timestamp"], df_origin["BBA"],
           s=70,
           color="#2E86DE",
           alpha=0.75,
           linewidths=0,
           label="FY-3D MERSI-II")

# -------- MODIS --------
ax.scatter(df_modis["timestamp"], df_modis["BBA"],
           s=40,
           marker="s",
           color="#27AE60",
           alpha=0.5,
           linewidths=0,
           label="MCD43A3",
           zorder=4)

#
ax.set_ylim(0, 1.0)
ax.set_xlim(start_date, end_date)

# 
ax.set_xlabel("")
ax.set_ylabel("Broadband Albedo", fontsize=16)

ax.yaxis.set_major_locator(plt.MultipleLocator(0.25))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

for label in ax.get_xticklabels():
    label.set_fontsize(16)
for label in ax.get_yticklabels():
    label.set_fontsize(16)

# 
ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))

ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

# legend
ax.legend(frameon=False, fontsize=16, loc="lower left",
          prop={'family': 'Arial', 'size': 16})

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.tight_layout()
plt.title('Dome C', fontsize=16)

# 
save_path = os.path.join(output_dir, "DomeC_albedo_300dpi.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"saved to: {save_path}")

plt.show()

print(f"\nData volume: MERSI-II={len(df_origin)}, MODIS={len(df_modis)}")


# 

stat_start = pd.to_datetime("2025-12-01")
stat_end   = pd.to_datetime("2026-03-01")

df_origin_s = df_origin[(df_origin["timestamp"] >= stat_start) & (df_origin["timestamp"] <= stat_end)].copy()
df_modis_s  = df_modis[(df_modis["timestamp"] >= stat_start) & (df_modis["timestamp"] <= stat_end)].copy()

# average
mean_mersi  = df_origin_s["BBA"].mean()
mean_modis  = df_modis_s["BBA"].mean()

# Daily-scale processing
df_origin_s["date"] = df_origin_s["timestamp"].dt.date
df_modis_s["date"] = df_modis_s["timestamp"].dt.date

mersi_daily  = df_origin_s.groupby("date")["BBA"].mean().reset_index()
modis_daily  = df_modis_s.groupby("date")["BBA"].mean().reset_index()

# Merge (keep only common dates)
df_daily = pd.merge(mersi_daily, modis_daily, on="date", how="inner")
df_daily.columns = ["date", "MERSI", "MODIS"]

#Statistical metrics

# Bias
bias = np.mean(df_daily["MERSI"] - df_daily["MODIS"])

# RMSE
rmse = np.sqrt(np.mean((df_daily["MERSI"] - df_daily["MODIS"])**2))

# R2
ss_res = np.sum((df_daily["MERSI"] - df_daily["MODIS"])**2)
ss_tot = np.sum((df_daily["MODIS"] - np.mean(df_daily["MODIS"]))**2)
r2 = 1 - ss_res / ss_tot

# MAD
mae = np.mean(np.abs(df_daily["MERSI"] - df_daily["MODIS"]))

# 
print("\n================ Statistical results =================")
print("time range: 2025-12-01 ~ 2026-03-01\n")

print(f"MERSI-II Mean BBA : {mean_mersi:.4f}")
print(f"MODIS Mean BBA    : {mean_modis:.4f}\n")

print("---- Daily matching（MERSI vs MODIS）----")
print(f"samples N   : {len(df_daily)}")
print(f"Bias       : {bias:.4f}")
print(f"RMSE       : {rmse:.4f}")
print(f"R2         : {r2:.4f}")
print(f"MAE        : {mae:.4f}")

MERSI-II/S3 OLCI SRF

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib
import os
from matplotlib.ticker import FuncFormatter

# Global plotting settings
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['axes.linewidth'] = 1.0
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['xtick.labelsize'] = 13
matplotlib.rcParams['ytick.labelsize'] = 13
matplotlib.rcParams['axes.labelsize'] = 13
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.unicode_minus'] = False

# Output path
output_dir = r"Figure"
os.makedirs(output_dir, exist_ok=True)

# ==========================
# Data paths
# ==========================
merged_fy3d_srf_path = r"D:\merged_FY3Dsrf_final.csv"
merged_olci_srf_path = r"D:\OLCI_SRF_Merged.csv"

# Read data
merged_fy3d_srf = pd.read_csv(merged_fy3d_srf_path)
merged_olci_srf = pd.read_csv(merged_olci_srf_path)

# Curve colors
fy3d_color = 'b'
olci_color = 'r'

# Specified bands
olci_bands_to_plot = [1, 6, 11, 12, 17, 21]
fy3d_bands_to_plot = [8, 11, 13, 14, 15, 19]

# Font size of the band numbers
band_number_fontsize = 14


def y_formatter(y, pos):
    """
    Format y-axis tick labels:
    display zero as 0, ±1 as ±1, and other values with one decimal place.
    """
    if abs(y) < 1e-6:
        return '0'
    elif abs(y - 1.0) < 1e-6:
        return '1'
    elif abs(y + 1.0) < 1e-6:
        return '-1'
    else:
        return f'{y:.1f}'


# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

# ==========================
# Plot OLCI band curves
# ==========================
for i, band in enumerate(olci_bands_to_plot):
    band_name = f'Oa{band:02d}'

    wavelengths = merged_olci_srf['Wavelength']
    responses = merged_olci_srf[band_name]

    # Retain positive spectral-response values
    valid_indices = responses[responses > 0].index
    valid_wavelengths = wavelengths.loc[valid_indices]
    valid_responses = responses.loc[valid_indices]

    # Plot OLCI curves below y=0
    ax.plot(
        valid_wavelengths,
        -valid_responses,
        color=olci_color,
        label='S3 OLCI' if i == 0 else ""
    )

    # Add band number at the maximum response
    if not valid_indices.empty:
        max_response_index = valid_responses.idxmax()

        ax.text(
            wavelengths.loc[max_response_index],
            -responses.loc[max_response_index],
            str(band),
            fontsize=band_number_fontsize,
            color=olci_color,
            ha='center',
            va='top'
        )

# ==========================
# Plot FY-3D MERSI-II curves
# ==========================
wavelengths = merged_fy3d_srf['Wavelength']

for i, band in enumerate(fy3d_bands_to_plot):
    band_name = f'FY3D_MERSI_SRF_CH{band:02d}_Pub.txt'
    responses = merged_fy3d_srf[band_name]

    # Retain positive spectral-response values
    valid_indices = responses[responses > 0].index
    valid_wavelengths = wavelengths.loc[valid_indices]
    valid_responses = responses.loc[valid_indices]

    ax.plot(
        valid_wavelengths,
        valid_responses,
        color=fy3d_color,
        label='FY-3D MERSI-II' if i == 0 else ""
    )

    # Add band number at the maximum response
    if not valid_indices.empty:
        max_response_index = valid_responses.idxmax()

        ax.text(
            wavelengths.loc[max_response_index],
            responses.loc[max_response_index],
            str(band),
            fontsize=band_number_fontsize,
            color=fy3d_color,
            ha='center',
            va='bottom'
        )

# ==========================
# Axis settings
# ==========================
ax.set_xlabel("Wavelength, nm", fontsize=13)
ax.set_ylabel("Reflectance", fontsize=13)
ax.set_title("Spectral Response Function", fontsize=13)

ax.set_xlim(350, 1100)

# Draw the y=0 dividing line
ax.axhline(
    0,
    color='gray',
    linewidth=1.5,
    linestyle='--'
)

# Tick settings
ax.tick_params(
    axis='both',
    which='both',
    direction='in'
)

# Format y-axis labels
ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))

# Legend
ax.legend(
    frameon=False,
    fontsize=13,
    loc='lower right',
    bbox_to_anchor=(1.0, -0.15),
    ncol=2,
    prop={
        'family': 'Arial',
        'size': 13
    }
)

# Border width
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.tight_layout()

# Save figure
save_path = os.path.join(output_dir, "srf.png")

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)

print(f"Image saved: {save_path}")

plt.show()